In [ ]:
import zipfile

with zipfile.ZipFile('perspective.zip') as zip_ref:
    zip_ref.extractall()

# Subtask 1

In [ ]:
import cv2
import os
from sklearn.metrics.pairwise import cosine_similarity

test_df_s1 = pd.read_csv('test_data.csv').dropna()

def get_features(row):
    img1 = cv2.imread(os.path.join('images', row['img1']))
    img2 = cv2.imread(os.path.join('images', row['img2']))
    img3 = cv2.imread(os.path.join('images', row['img3']))

    for img in [img1, img2, img3]:
        img = cv2.resize(img, (32, 32), cv2.INTER_AREA)

    return img1.reshape(1, -1), img2.reshape(1, -1), img3.reshape(1, -1)

def compare_images(img1, img2, img3):
    sim12 = cosine_similarity(img1, img2)
    sim23 = cosine_similarity(img2, img3)
    sim13 = cosine_similarity(img1, img3)

    max_sim = max(sim12, sim23, sim13)
    if max_sim == sim12:
        return 3
    elif max_sim == sim23:
        return 1
    else:
        return 2

answer_s1 = []

for index in test_df_s1.index:
    img1, img2, img3 = get_features(test_df_s1.iloc[index])
    answer_s1.append(compare_images(img1, img2, img3))

print('Done')

# Subtask 2

In [ ]:
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
import numpy as np

train_df = pd.read_csv('train_data.csv')
test_df_s2 = pd.read_csv('test_data.csv')
test_df_s2 = test_df_s2[test_df_s2['subtaskID'] == 2]

device = 'cuda' if torch.cuda.is_available() else 'cpu'

device

In [ ]:
class AngleDataset(Dataset):
    def __init__(self, df, transformer, has_labels):
        self.df = df
        self.transformer = transformer
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        if self.has_labels:
            img = Image.open(os.path.join('images', row['image'])).convert('RGB')
            target = float(row['angle'])
            return self.transformer(img), np.float32(target)
        else:
            img = Image.open(os.path.join('images', row['img1'])).convert('RGB')
            return self.transformer(img)

from torchvision import transforms

train_transformer = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

train_ds = AngleDataset(train_df, train_transformer, has_labels=True)
test_ds = AngleDataset(test_df_s2, train_transformer, has_labels=False)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
def get_model():
    model = torchvision.models.resnet18(weights=None)
    model.fc = nn.LazyLinear(1)
    return model

model = get_model().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

torch.cuda.empty_cache()
for epoch in range(1, 76):
    model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets.unsqueeze(-1))
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /=  len(train_loader)
    scheduler.step(train_loss)
    print(f'Epoch {epoch:2d} | Loss: {train_loss:.4f}')

In [ ]:
model.eval()
answer_s2 = []
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        batch_preds = model(images)
        answer_s2.extend(batch_preds.cpu().numpy())

answer_s2 = np.clip(answer_s2, 0, 360)

#The angles are in steps of 5 degree
#This line gets the preds to the nearest multiple of 5 to reduce MAE
answer_s2 = (np.round(answer_s2 / 5) * 5).astype(int)

In [ ]:
test_df = pd.read_csv('test_data.csv')

output_df = pd.DataFrame({
    'subtaskID': test_df['subtaskID'],
    'datapointID':test_df['datapointID'],
    'answer':[str(x) for x in answer_s1] + [x.item() for x in answer_s2]
})

output_df.to_csv('submission.csv', index=False)
output_df.head()